# 推理服务：Prefill、Decode、KV cache 与成本

解释推理服务的时间和内存分解，计算 KV 基线与每成功任务成本，用指标定位瓶颈。

Python 3.10+ / CPU / 无密钥。固定教学 fixture，不调用模型或服务。先读对应正文，预测成功与失败，再运行并修改一个边界。

In [ ]:
import math
def kv_bytes(layers,kv_heads,head_dim,tokens,bytes_each=2):
    if any(type(x) is not int or x<=0 for x in [layers,kv_heads,head_dim,tokens,bytes_each]):
        raise ValueError('positive integer dimensions required')
    return 2*layers*kv_heads*head_dim*tokens*bytes_each
size=kv_bytes(32,8,128,4096)
assert size==536870912  # 512 MiB，教学架构
costs=[1.0,2.0,3.0,4.0];success=[True,False,True,False]
per_success=sum(costs)/sum(success)
assert per_success==5.0  # 失败费用也计入
latencies=[1,2,3,4,20]
p95=sorted(latencies)[math.ceil(.95*len(latencies))-1]
assert p95==20  # 本例采用 nearest-rank，非所有工具的默认插值法
checks={'kv_baseline':True,'failed_cost_included':True,'tail_latency':True}
print(size//(1024**2),per_success,p95)

## 采样的支持集
先预测top-p跨阈值token、温度与平局规则。概率是原创fixture，不是模型输出。


In [ ]:
import math
def probabilities(values):
    if not values or any(type(x) not in (int,float) or not math.isfinite(x) or x<0 for x in values):
        raise ValueError('finite nonnegative probabilities')
    if not math.isclose(sum(values),1.,abs_tol=1e-12): raise ValueError('probabilities must sum to one')
    return list(values)
def nucleus(values,threshold):
    values=probabilities(values)
    if type(threshold) not in (int,float) or not math.isfinite(threshold) or not 0<threshold<=1:
        raise ValueError('top-p in (0,1]')
    ordered=sorted(range(len(values)),key=lambda i:(-values[i],i));kept=[];mass=0.
    for i in ordered:
        if values[i]==0: break
        kept.append(i);mass+=values[i]
        if mass>=threshold:break
    return [v/mass if i in kept else 0. for i,v in enumerate(values)]
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(nucleus([.6,.3,.1],.8),[2/3,1/3,0.]))
assert nucleus([.6,.3,.1],.6)==[1.,0.,0.]
assert nucleus([.5,.5,0.],1)==[.5,.5,0.]
for invalid in [( [],.8),([.2,.2],.8),([.6,.3,.1],0),([math.nan,1.],.8)]:
    try:nucleus(*invalid)
    except ValueError:pass
    else:raise AssertionError('invalid sampling input accepted')
checks['sampling_support']=True
print('top-p=.8:',nucleus([.6,.3,.1],.8))

## 穷举投机采样的一位置概率质量
拒绝后用(p−q)+，验证p=q、支持集不交和非法概率。不是多步引擎或GPU吞吐实测。


In [ ]:
def speculative_mass(target,draft):
    target,draft=probabilities(target),probabilities(draft)
    if len(target)!=len(draft):raise ValueError('same vocabulary required')
    accepted=[min(p,q) for p,q in zip(target,draft)]
    rejected=1-sum(accepted)
    residual=[max(0,p-q) for p,q in zip(target,draft)];total=sum(residual)
    if total==0:return accepted
    return [a+rejected*r/total for a,r in zip(accepted,residual)]
p=[.6,.3,.1];q=[.2,.5,.3]
for p_case,q_case in [(p,q),(p,p),([1.,0.],[0.,1.]),([.5,.5],[1.,0.])]:
    actual=speculative_mass(p_case,q_case)
    assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(actual,p_case))
wrong=[min(a,b)+(1-sum(min(c,d) for c,d in zip(p,q)))*a for a,b in zip(p,q)]
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(wrong,[.44,.42,.14]))
assert not all(math.isclose(a,b) for a,b in zip(wrong,p))
try:speculative_mass([1.],[.5,.5])
except ValueError:pass
else:raise AssertionError('different vocabulary accepted')
def ideal_rate(alpha,drafts,verify_ms,draft_ms):
    if not 0<=alpha<=1 or type(drafts) is not int or drafts<1 or verify_ms<=0 or draft_ms<0:
        raise ValueError('rate dimensions')
    return sum(alpha**j for j in range(drafts+1))/(verify_ms+drafts*draft_ms)
fast=ideal_rate(.6,4,1.3,.1);slow=ideal_rate(.1,4,1.3,.1)
assert math.isclose(fast,2.3056/1.7) and fast>1 and slow<1
checks['speculative_distribution']=True
print('target/reconstructed/wrong:',p,speculative_mass(p,q),wrong)
print('ideal token/ms, not measured:',fast,slow)

## 块分配与量化的算式
块槽不是总显存；int8是固定scale教学，不是GPTQ/AWQ、FP8或实际量化kernel。


In [ ]:
def allocated_slots(lengths,block):
    if type(block) is not int or block<=0 or not lengths or any(type(n) is not int or n<=0 for n in lengths):
        raise ValueError('positive lengths/block')
    return sum(((n+block-1)//block)*block for n in lengths)
assert allocated_slots([17,31],16)==64
assert allocated_slots([33],16)-allocated_slots([17],16)==16
for lengths,block in [([0],16),([17],0),([True],16)]:
    try:allocated_slots(lengths,block)
    except ValueError:pass
    else:raise AssertionError('invalid allocation accepted')
checks['kv_block_allocation']=True
def int8_scalar(x,scale):
    if any(type(v) not in (int,float) or not math.isfinite(v) for v in [x,scale]) or scale<=0:
        raise ValueError('finite value and positive scale')
    # Python round is ties-to-even; real kernels must document their rounding.
    q=max(-127,min(127,round(x/scale)))
    return q,q*scale
assert int8_scalar(12.8,.1)[0]==127
assert math.isclose(int8_scalar(12.8,.1)[1],12.7)
assert int8_scalar(-12.8,.1)[0]==-127
for value,scale in [(1.,0.),(math.inf,.1)]:
    try:int8_scalar(value,scale)
    except ValueError:pass
    else:raise AssertionError('invalid quantization accepted')
checks['quantization_clipping']=True
print('slots/used/wasted:',64,48,16,'quantized:',int8_scalar(12.8,.1))

## 记录与继续

在下一格 my_explanation 写下预测、改动与实际观察；运行后下载 mechanism_report.json，导回本篇笔记。下一次先独立解释边界，再对照旧记录。

In [ ]:
import json
from pathlib import Path
my_explanation = ''
report={'experiment':'lesson-inference-serving', 'scope':'stdlib fixtures; no model/service', 'checks':checks, 'my_explanation':my_explanation}
Path('mechanism_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps(report,ensure_ascii=False,indent=2))
